# 📊 Google Colab + Kaggle + SQL
## Opérations CRUD Complètes avec Données Réelles

---

### 🎯 Objectifs de ce Notebook
- Télécharger un dataset Kaggle
- Décrire la structure de la base de données
- Tester des requêtes INSERT
- Tester des requêtes SELECT
- Tester des requêtes UPDATE
- Tester des requêtes DELETE

**Temps estimé:** 15-20 minutes

---

## Section 1: Installation des Dépendances

Installez automatiquement tous les packages nécessaires.

In [ ]:
# Installation des dépendances
import subprocess
import sys
import os

# Désactiver temporairement l'authentification automatique lors de l'import
os.environ['KAGGLE_CONFIG_DIR'] = os.path.expanduser('~/.kaggle')

packages = ['kaggle', 'pandas', 'numpy', 'matplotlib', 'seaborn']

print("Installation des dépendances...\n")

for package in packages:
    try:
        if package == 'kaggle':
            # Pour kaggle, on vérifie via pip ou import sans déclencher l'authentification immédiate
            import importlib.util
            spec = importlib.util.find_spec(package)
            if spec is None:
                raise ImportError
        else:
            __import__(package)
        print(f"✓ {package} est déjà installé")
    except (ImportError, NameError):
        print(f"Installation de {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installé")

print("\n✅ Toutes les dépendances sont prêtes!")

## Section 2: Configuration Kaggle (optionnelle)

Le dataset Titanic est téléchargé plus bas via une URL publique : **l'API Kaggle n'est pas obligatoire**.
Si vous voulez quand même l'utiliser, ajoutez votre clé dans les *Secrets* Colab (icône 🔑 à gauche) sous les noms `KAGGLE_USERNAME` et `KAGGLE_KEY` — ne collez jamais la clé en clair dans le notebook.

In [ ]:
import os

# Lecture sécurisée des identifiants Kaggle depuis les Secrets Colab (aucune clé en clair)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    print("✓ Authentification Kaggle réussie")
except Exception as e:
    print(f"ℹ️  Kaggle non configuré ({type(e).__name__}) — pas grave, on utilise l'URL publique.")

## Section 3: Connexion Google Drive

Montez Google Drive pour sauvegarder les données (hors Colab, un dossier local est utilisé).

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive monté")
    drive_data_dir = '/content/drive/MyDrive/Colab Datasets'
except ImportError:
    drive_data_dir = os.path.abspath('Colab Datasets')
    print("ℹ️  Hors Colab : utilisation d'un dossier local")

os.makedirs(drive_data_dir, exist_ok=True)
print(f"✓ Dossier de travail : {drive_data_dir}")

## Section 4: Téléchargement du Dataset Kaggle

Téléchargez le dataset Titanic depuis Kaggle.

In [ ]:
# Téléchargement du dataset Titanic via une URL publique
import urllib.request

dataset_name = 'titanic'
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
destination_file = os.path.join(drive_data_dir, 'train.csv')

print("Téléchargement du dataset depuis l'URL publique...\n")
urllib.request.urlretrieve(url, destination_file)
print(f"✓ Dataset téléchargé : {destination_file}\n")

print("Fichiers dans le dossier :")
for file in os.listdir(drive_data_dir):
    file_path = os.path.join(drive_data_dir, file)
    if os.path.isfile(file_path):
        print(f"  - {file} ({os.path.getsize(file_path):,} bytes)")

## Section 5: Création de la Base de Données SQLite

In [ ]:
import sqlite3
import pandas as pd

# Chemins des fichiers
db_path = os.path.join(drive_data_dir, 'titanic_database.db')
train_csv = os.path.join(drive_data_dir, 'train.csv')

# Supprimer l'ancienne base si elle existe
if os.path.exists(db_path):
    os.remove(db_path)
    print("Ancienne base de données supprimée")

# Créer la connexion
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"✓ Base de données créée: {db_path}\n")

# Charger les données
df = pd.read_csv(train_csv)
print(f"✓ Données chargées: {len(df)} lignes, {len(df.columns)} colonnes\n")

# Créer la table
table_name = 'passengers'
df.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"✓ Table '{table_name}' créée dans la base de données")

## Section 6: Description de la Structure de la Base de Données

In [ ]:
# Informations sur la table
cursor.execute(f"PRAGMA table_info({table_name})")
columns_info = cursor.fetchall()

print("\n📋 STRUCTURE DE LA TABLE")
print("=" * 80)

columns_df = pd.DataFrame(columns_info, columns=['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk'])
print(columns_df[['name', 'type', 'notnull']].to_string(index=False))

# Statistiques
print("\n\n📊 STATISTIQUES")
print("=" * 80)

cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
total_rows = cursor.fetchone()[0]
print(f"Total des lignes: {total_rows:,}")

# Analyse détaillée
print("\n📈 ANALYSE PAR COLONNE")
print("=" * 80)

for col_info in columns_info[:8]:  # Afficher les 8 premières colonnes
    col_name = col_info[1]
    col_type = col_info[2]

    cursor.execute(f"SELECT COUNT(DISTINCT {col_name}) FROM {table_name} WHERE {col_name} IS NOT NULL")
    distinct_count = cursor.fetchone()[0]

    cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE {col_name} IS NULL")
    null_count = cursor.fetchone()[0]

    print(f"\n  {col_name} ({col_type}):")
    print(f"    Valeurs uniques: {distinct_count}")
    print(f"    Valeurs NULL: {null_count}")

    if col_type in ['INTEGER', 'REAL']:
        cursor.execute(f"SELECT MIN({col_name}), MAX({col_name}), AVG({col_name}) FROM {table_name}")
        min_val, max_val, avg_val = cursor.fetchone()
        if min_val is not None:
            print(f"    Min: {min_val}, Max: {max_val}, Moyenne: {avg_val:.2f}")

## Aperçu des Données

In [ ]:
from IPython.display import display

print("\n📋 PREMIÈRES LIGNES DE LA TABLE")
print("=" * 80 + "\n")

query = f"SELECT * FROM {table_name} LIMIT 5"
result_df = pd.read_sql_query(query, conn)
display(result_df)

---

# 📝 OPÉRATIONS SQL

## Section 7: REQUÊTES INSERT

### 1. Insert Simple

In [ ]:
print("\n1️⃣  INSERT SIMPLE")
print("=" * 80)

new_passenger = {
    'PassengerId': 892,
    'Survived': 1,
    'Pclass': 1,
    'Name': 'Claude, Monsieur',
    'Sex': 'male',
    'Age': 35.0,
    'SibSp': 1,
    'Parch': 1,
    'Ticket': 'AI001',
    'Fare': 512.33,
    'Cabin': 'A123',
    'Embarked': 'S'
}

insert_query = f"""
INSERT INTO {table_name}
(PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

values = tuple(new_passenger.values())
cursor.execute(insert_query, values)
conn.commit()

print("\n✓ Requête INSERT exécutée avec succès!")
print(f"\nDonnées insérées:")
for key, value in new_passenger.items():
    print(f"  {key}: {value}")

# Vérification
cursor.execute(f"SELECT * FROM {table_name} WHERE PassengerId = 892")
inserted_row = cursor.fetchone()
print(f"\n✓ Vérification: Ligne retrouvée dans la base de données")

### 2. Insert Multiple (Bulk Insert)

In [ ]:
print("\n2️⃣  INSERT MULTIPLE (BULK INSERT)")
print("=" * 80)

new_passengers = [
    (893, 0, 3, 'Dupont, Monsieur', 'male', 25.0, 0, 0, 'AI002', 7.75, None, 'C'),
    (894, 1, 1, 'Lefevre, Madame', 'female', 28.0, 1, 1, 'AI003', 151.86, 'B5', 'S'),
    (895, 0, 2, 'Martin, Monsieur', 'male', 45.0, 0, 0, 'AI004', 10.50, None, 'Q'),
]

bulk_insert_query = f"""
INSERT INTO {table_name}
(PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

cursor.executemany(bulk_insert_query, new_passengers)
conn.commit()

print("✓ Insertion en masse réussie!")
print(f"✓ {len(new_passengers)} passagers insérés\n")

print("Détail des insertions:")
for i, passenger in enumerate(new_passengers, 1):
    print(f"  Passager {i}: PassengerId={passenger[0]}, Name={passenger[3]}, Age={passenger[5]}")

# Vérifier
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
total_after = cursor.fetchone()[0]
print(f"\nTotal des lignes après insertion: {total_after:,}")

---

## Section 8: REQUÊTES SELECT

### 1. SELECT Simple

In [ ]:
print("\n1️⃣  SELECT SIMPLE")
print("=" * 80 + "\n")

query = f"SELECT * FROM {table_name} WHERE PassengerId >= 890 LIMIT 5"
print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)

### 2. SELECT avec Filtres

In [ ]:
print("\n2️⃣  SELECT AVEC FILTRES")
print("=" * 80 + "\n")

query = f"""
SELECT Name, Age, Pclass, Fare
FROM {table_name}
WHERE Age > 30 AND Survived = 1
ORDER BY Age DESC
LIMIT 10
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
print(f"Résultat: {len(result_df)} passagers trouvés\n")
display(result_df)

### 3. SELECT avec Agrégation

In [ ]:
print("\n3️⃣  SELECT AVEC AGRÉGATION")
print("=" * 80 + "\n")

query = f"""
SELECT
    Survived,
    COUNT(*) as nb_passagers,
    ROUND(AVG(Age), 2) as age_moyen,
    ROUND(AVG(Fare), 2) as fare_moyen
FROM {table_name}
GROUP BY Survived
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)

### 4. SELECT Statistiques par Classe

In [ ]:
print("\n4️⃣  STATISTIQUES PAR CLASSE")
print("=" * 80 + "\n")

query = f"""
SELECT
    Pclass as classe,
    COUNT(*) as nb_passagers,
    SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) as survivants,
    ROUND(100.0 * SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as taux_survie_pct
FROM {table_name}
WHERE Pclass IS NOT NULL
GROUP BY Pclass
ORDER BY Pclass
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)

### 5. SELECT Top 10 (Les plus riches)

In [ ]:
print("\n5️⃣  TOP 10 PASSAGERS LES PLUS RICHES")
print("=" * 80 + "\n")

query = f"""
SELECT
    Name,
    Age,
    Sex,
    Fare,
    Survived
FROM {table_name}
WHERE Fare IS NOT NULL
ORDER BY Fare DESC
LIMIT 10
"""

print(f"Requête: {query}\n")
result_df = pd.read_sql_query(query, conn)
display(result_df)

---

## Section 9: REQUÊTES UPDATE

### 1. Update Simple

In [ ]:
print("\n1️⃣  UPDATE SIMPLE")
print("=" * 80 + "\n")

# Avant
cursor.execute(f"SELECT PassengerId, Name, Age, Fare FROM {table_name} WHERE PassengerId = 892")
before = cursor.fetchone()
print("Avant modification:")
print(f"  PassengerId: {before[0]}, Name: {before[1]}, Age: {before[2]}, Fare: {before[3]}")

# Update
update_query = f"""
UPDATE {table_name}
SET Age = 40, Fare = 550.00
WHERE PassengerId = 892
"""

print(f"\nRequête UPDATE: {update_query}")
cursor.execute(update_query)
conn.commit()
print("✓ Modification réussie!")

# Après
cursor.execute(f"SELECT PassengerId, Name, Age, Fare FROM {table_name} WHERE PassengerId = 892")
after = cursor.fetchone()
print("\nAprès modification:")
print(f"  PassengerId: {after[0]}, Name: {after[1]}, Age: {after[2]}, Fare: {after[3]}")

### 2. Update Multiple Lignes

In [ ]:
print("\n2️⃣  UPDATE PLUSIEURS LIGNES")
print("=" * 80 + "\n")

# Avant
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Age IS NULL AND Pclass = 1")
before_count = cursor.fetchone()[0]

update_query = f"""
UPDATE {table_name}
SET Age = 30
WHERE Age IS NULL AND Pclass = 1
"""

print(f"Requête UPDATE: {update_query}")
print(f"Lignes affectées avant: {before_count}")

cursor.execute(update_query)
rows_affected = cursor.rowcount
conn.commit()
print(f"✓ {rows_affected} lignes mises à jour")

# Après
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Age = 30 AND Pclass = 1")
after_count = cursor.fetchone()[0]
print(f"Lignes avec Age=30 et Pclass=1: {after_count}")

### 3. Update Conditionnel Complexe

In [ ]:
print("\n3️⃣  UPDATE CONDITIONNEL COMPLEXE")
print("=" * 80 + "\n")

update_query = f"""
UPDATE {table_name}
SET Fare = Fare * 1.1
WHERE Pclass = 1 AND Fare IS NOT NULL
"""

print(f"Requête UPDATE: {update_query}")
print("(Augmentation de 10% des tarifs pour la première classe)\n")

cursor.execute(update_query)
conn.commit()
print(f"✓ {cursor.rowcount} lignes mises à jour")

# Aperçu
query = f"""
SELECT Name, Pclass, Fare FROM {table_name}
WHERE Pclass = 1 AND Fare IS NOT NULL
LIMIT 5
"""
result_df = pd.read_sql_query(query, conn)
print("\nAperçu des premières classes mises à jour:")
display(result_df)

---

## Section 10: REQUÊTES DELETE

### 1. Delete Simple

In [ ]:
print("\n1️⃣  DELETE SIMPLE")
print("=" * 80 + "\n")

# Avant suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE PassengerId = 893")
exists_before = cursor.fetchone()[0]
print(f"Ligne avec PassengerId=893 existe avant suppression: {bool(exists_before)}")

# Supprimer
delete_query = f"DELETE FROM {table_name} WHERE PassengerId = 893"
print(f"\nRequête DELETE: {delete_query}")

cursor.execute(delete_query)
conn.commit()
print(f"✓ {cursor.rowcount} ligne supprimée")

# Après suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE PassengerId = 893")
exists_after = cursor.fetchone()[0]
print(f"Ligne avec PassengerId=893 existe après suppression: {bool(exists_after)}")

### 2. Delete Multiple Lignes

In [ ]:
print("\n2️⃣  DELETE PLUSIEURS LIGNES")
print("=" * 80 + "\n")

# Insérer des lignes de test
test_passengers = [
    (896, 0, 3, 'Test1, Monsieur', 'male', 20.0, 0, 0, 'TEST1', 5.00, None, 'C'),
    (897, 1, 3, 'Test2, Madame', 'female', 22.0, 0, 0, 'TEST2', 7.75, None, 'S'),
    (898, 0, 3, 'Test3, Monsieur', 'male', 18.0, 0, 0, 'TEST3', 6.50, None, 'Q'),
]

cursor.executemany(bulk_insert_query, test_passengers)
conn.commit()
print("✓ 3 lignes de test insérées")

# Avant suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Ticket LIKE 'TEST%'")
before_delete = cursor.fetchone()[0]
print(f"Lignes de test avant suppression: {before_delete}")

# Supprimer
delete_query = f"DELETE FROM {table_name} WHERE Ticket LIKE 'TEST%'"
print(f"\nRequête DELETE: {delete_query}")

cursor.execute(delete_query)
conn.commit()
print(f"✓ {cursor.rowcount} lignes supprimées")

# Après suppression
cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE Ticket LIKE 'TEST%'")
after_delete = cursor.fetchone()[0]
print(f"Lignes de test après suppression: {after_delete}")

---

## Section 11: Résumé et Statistiques Finales

In [ ]:
print("\n" + "=" * 80)
print("RÉSUMÉ ET STATISTIQUES FINALES")
print("=" * 80)

cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
final_count = cursor.fetchone()[0]

print(f"\n📊 RÉSUMÉ DES OPÉRATIONS:")
print("-" * 80)
print(f"✓ Base de données créée: {db_path}")
print(f"✓ Dataset téléchargé: {dataset_name}")
print(f"✓ Table créée: {table_name}")
print(f"✓ Total des lignes finales: {final_count:,}")
print(f"✓ Total des colonnes: {len(columns_info)}")

print(f"\n📈 OPÉRATIONS EFFECTUÉES:")
print("-" * 80)
print(f"  • INSERT: 4 passagers insérés")
print(f"  • SELECT: 5 requêtes de sélection exécutées")
print(f"  • UPDATE: 3 requêtes de mise à jour exécutées")
print(f"  • DELETE: 2 requêtes de suppression exécutées")

print(f"\n📋 STATISTIQUES FINALES:")
print("-" * 80)

query = f"""
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) as survivants,
    ROUND(100.0 * SUM(CASE WHEN Survived = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as taux_survie,
    ROUND(AVG(Age), 2) as age_moyen,
    ROUND(AVG(Fare), 2) as fare_moyen
FROM {table_name}
WHERE Age IS NOT NULL
"""

result_df = pd.read_sql_query(query, conn)
display(result_df)

# Fermer la connexion
conn.close()
print("\n✓ Connexion à la base de données fermée")
print("\n" + "=" * 80)
print("✅ SCRIPT TERMINÉ AVEC SUCCÈS!")
print("=" * 80)

---

## 📚 Ressources Supplémentaires

### Fichiers Générés
Vos données sont sauvegardées ici :
```
Google Drive > Colab Datasets > titanic_database.db
```

### Documentation
- **README_COLAB.md** : Guide complet
- **sql_queries_examples.py** : 30+ requêtes SQL
- **20_Exercices_SQL.docx** : Exercices pratiques

### Prochaines Étapes
1. Modifiez les données selon vos besoins
2. Testez d'autres datasets (Iris, Housing, Adult)
3. Explorez les requêtes SQL avancées
4. Créez vos propres analyses

### Datasets Recommandés
- Titanic (893 lignes) : Débutant
- Iris (150 lignes) : Classification simple
- Housing (20K lignes) : Régression
- Adult (32K lignes) : Données complexes

---

**Bon learning! 🚀**

---
# 🧩 Section 12 : Exercice — 10 questions SQL

On recharge une table **`titanic`** propre à partir de `train.csv` (les 891 passagers d'origine), pour que les INSERT/UPDATE/DELETE précédents ne faussent pas les réponses.

In [ ]:
import sqlite3, pandas as pd, os
from IPython.display import display

conn = sqlite3.connect(os.path.join(drive_data_dir, 'titanic_database.db'))
pd.read_csv(os.path.join(drive_data_dir, 'train.csv')).to_sql('titanic', conn, if_exists='replace', index=False)

def q(titre, sql):
    print(f"\n{titre}\n" + "=" * 80)
    print(sql.strip())
    display(pd.read_sql_query(sql, conn))

print("✓ Table 'titanic' prête :", pd.read_sql_query("SELECT COUNT(*) AS n FROM titanic", conn).n[0], "lignes")

In [ ]:
q("1. Top 5 passagers les plus âgés ayant survécu", """
SELECT Name, Age, Sex, Pclass
FROM titanic
WHERE Survived = 1 AND Age IS NOT NULL
ORDER BY Age DESC
LIMIT 5
""")

In [ ]:
q("2. Tarif moyen par classe de voyage", """
SELECT Pclass AS classe,
       COUNT(*) AS nb_passagers,
       ROUND(AVG(Fare), 2) AS tarif_moyen
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""")

In [ ]:
q("3. Taux de survie par sexe", """
SELECT Sex,
       COUNT(*) AS nb_passagers,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
GROUP BY Sex
""")

In [ ]:
q("4. Passagers de première classe embarqués à Southampton", """
SELECT PassengerId, Name, Sex, Age, Fare
FROM titanic
WHERE Pclass = 1 AND Embarked = 'S'
ORDER BY Name
""")

In [ ]:
q("4b. Nombre de passagers de 1re classe de Southampton", """
SELECT COUNT(*) AS nb_1re_classe_southampton
FROM titanic
WHERE Pclass = 1 AND Embarked = 'S'
""")

In [ ]:
q("5. Statistiques d'âge par classe", """
SELECT Pclass AS classe,
       COUNT(Age) AS ages_connus,
       MIN(Age) AS age_min,
       MAX(Age) AS age_max,
       ROUND(AVG(Age), 2) AS age_moyen
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""")

In [ ]:
q("6. Passagers avec famille à bord (SibSp + Parch > 0)", """
SELECT COUNT(*) AS avec_famille,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
WHERE SibSp + Parch > 0
""")

In [ ]:
q("6b. Détail des passagers avec famille (10 plus grandes familles)", """
SELECT Name, SibSp, Parch, SibSp + Parch AS taille_famille, Survived
FROM titanic
WHERE SibSp + Parch > 0
ORDER BY taille_famille DESC, Name
LIMIT 10
""")

In [ ]:
q("7. Enfants de moins de 10 ans", """
SELECT COUNT(*) AS nb_enfants,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
WHERE Age < 10
""")

In [ ]:
q("8. Taux de survie par sexe + classe", """
SELECT Sex, Pclass AS classe,
       COUNT(*) AS nb_passagers,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
""")

In [ ]:
q("9. Passagers sans numéro de cabine", """
SELECT COUNT(*) AS sans_cabine,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM titanic), 2) AS pct_total
FROM titanic
WHERE Cabin IS NULL OR Cabin = ''
""")

In [ ]:
q("10. Top 10 des tarifs les plus élevés", """
SELECT Name, Pclass, Fare, Survived
FROM titanic
ORDER BY Fare DESC
LIMIT 10
""")

In [ ]:
conn.close()
print('✓ Exercice terminé')